# TENSORIX v2: final evaluation (evaluation owner)

Produces every number and figure for the final report and slides, from the finished checkpoints on Drive:

| Cell | What it measures | Runtime | Time (T4, estimate) |
|---|---|---|---|
| 1 | Setup | any | 2 min |
| 2 | Configuration (edit here) | any | - |
| 3 | `evaluate.py`: C4 / TinyStories perplexity, size, memory, latency for every pretrained run + teacher, with the log.csv consistency check | **GPU** | ~2 min per model |
| 4 | `bench_mc.py`: HellaSwag (10,042) and ARC-Easy (2,376) zero-shot accuracy | **GPU** | ~5 min per student, ~15 min teacher |
| 5 | Held-out Dolly prompts (from the SFT validation split) and answers from every SFT model, same seed per prompt | **GPU** | ~10 min |
| 6 | Blind pairwise LLM judging (both orders, sign test) + a blind sheet for a human check | CPU is fine | ~1 h per 300 prompts at 10 requests/min |
| 7 | `final_report.py`: RESULTS.md, final_metrics.json and all figures | CPU | 1 min |

Rules: run cells **one at a time, top to bottom**; never while a training run uses the GPU. Every cell skips work whose output already exists on Drive, so after a disconnect just rerun Cell 1, Cell 2 and the cell you were on.
Cells 6 and 7 do not need a GPU: to save quota, disconnect the GPU after Cell 5, switch the runtime to CPU, and run Cells 1, 2, 6, 7.

Outputs go to `MyDrive/tensorix/v2/results/final/`. Cell 7 also writes `final_results.zip`: unzip it into `results/final/` in the repo and commit it on a branch.

## Cell 1: Setup (run first in every session)

In [ ]:
import os, sys, json, glob, shutil, subprocess, torch
BRANCH = "main"
REPO   = "https://github.com/sunidhishekar-create/tensorix-kd-slm.git"
DRIVE  = "/content/drive/MyDrive/tensorix/v2"
CK     = f"{DRIVE}/checkpoints"
OUT    = f"{DRIVE}/results/final"

from google.colab import drive
drive.mount("/content/drive")

%cd /content
if not os.path.exists("/content/tensorix-kd-slm"):
    !git clone {REPO}
%cd /content/tensorix-kd-slm
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
!git log --oneline -3
!pip install -q -r requirements.txt google-genai

os.makedirs("/content/data", exist_ok=True)
for f in ("val_c4.bin", "val_ts.bin", "meta.json"):          # train.bin is NOT needed for evaluation
    if not os.path.exists(f"/content/data/{f}"):
        shutil.copy(f"{DRIVE}/data/{f}", f"/content/data/{f}")
for sub in ("eval", "bench", "logs", "judge"):
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)

def run(cmd):
    # Run a command and stream its output into the notebook; stop the cell if it fails.
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode}: {' '.join(cmd)}")

GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "NONE (fine for Cells 6 and 7 only)")

## Cell 2: Configuration (the only cell you normally edit)
Runs that are missing on Drive are skipped automatically (for example `kd_full_T2` until it exists).

In [ ]:
TEACHER = "HuggingFaceTB/SmolLM2-360M"
PRETRAIN_RUNS = ["baseline_short", "kd_short_T2", "kd_short_T4", "kd_short_T1", "baseline_full", "kd_full_T2"]

# instruction-tuned models: short name used in tables -> run folder in checkpoints/
SFT_RUNS = {"sft_base_short": "sft_dolly_baseline_short_v1",   # SHORT tier, no KD   (matched control)
            "sft_kd_T2":      "sft_dolly_kd_short_T2_v1",      # SHORT tier, KD T=2
            "sft_base_full":  "sft_dolly_baseline_v3"}         # FULL tier, no KD (4x the pretraining tokens)
SFT_REF_SPLIT = f"{CK}/sft_dolly_baseline_v3/split.json"       # the held-out prompts must match this split

N_PROMPTS = 300              # held-out Dolly prompts (of 592) used for generation and judging
GEN = dict(temperature=0.7, top_k=40, max_new_tokens=120, seed=0)   # max_new_tokens 120 as in the SFT notebooks

JUDGE_MODEL = "gemma-4-26b-a4b-it"   # through the Gemini API (key in Colab Secrets as GEMINI_API_KEY)
JUDGE_RPM = 10
# (pair, number of prompts, forced choice?). The first pair is the main result: same tier, only KD differs.
JUDGE_JOBS = [("sft_base_short:sft_kd_T2", N_PROMPTS, True),
              ("sft_base_short:sft_kd_T2", N_PROMPTS, False),
              ("sft_base_full:sft_kd_T2", 100, True)]
HUMAN_SHEET = 40             # prompts in the blind sheet for a human check (0 = skip)

FORCE = False                # True = recompute even if the output file already exists

runs = [r for r in PRETRAIN_RUNS if os.path.exists(f"{CK}/{r}/best.pt")]
sft = {k: v for k, v in SFT_RUNS.items() if os.path.exists(f"{CK}/{v}/best.pt")}
print("pretrained runs found:", runs, "| missing:", [r for r in PRETRAIN_RUNS if r not in runs])
print("SFT runs found:", sft, "| missing:", [k for k in SFT_RUNS if k not in sft])

## Cell 3: Perplexity, size, memory and latency (`src/evaluate.py`) + consistency check
Expected: every student matches the last row of its `log.csv` to about 3 decimals (stop and tell Sunidhi if not); the teacher scores C4 about 3.047 and TinyStories about 1.907.

In [ ]:
assert GPU, "Cell 3 needs a GPU runtime"
%cd /content/tensorix-kd-slm
todo = [(r, ["--ckpt", f"{CK}/{r}/best.pt"]) for r in runs] + [("teacher_base", ["--hf_model", TEACHER])]
for name, src in todo:
    if os.path.exists(f"{OUT}/eval/{name}.json") and not FORCE:
        print("already done:", name); continue
    run([sys.executable, "src/evaluate.py", "--data_dir", "/content/data", "--name", name,
         "--out_dir", f"{OUT}/eval"] + src)

import csv, pandas as pd
rows = []
for name, _ in todo:
    e = json.load(open(f"{OUT}/eval/{name}.json"))
    check = "-"
    log = f"{CK}/{name}/log.csv"
    if os.path.exists(log):
        last = list(csv.DictReader(open(log)))[-1]
        d = max(abs(float(last["val_c4"]) - e["val_c4_loss"]), abs(float(last["val_ts"]) - e["val_ts_loss"]))
        check = f"OK (max diff {d:.4f})" if d < 2e-3 else f"MISMATCH (max diff {d:.4f})"
        shutil.copy(log, f"{OUT}/logs/{name}_log.csv")          # training curves for the figures
    rows.append({"model": name, "step": e.get("step"), "C4 loss": e["val_c4_loss"], "C4 ppl": e["val_c4_ppl"],
                 "TS loss": e["val_ts_loss"], "TS ppl": e["val_ts_ppl"], "params (M)": e["params_m"],
                 "peak mem (MB)": e["peak_mem_mb"], "latency (ms)": e["latency_ms_1x256"], "log.csv check": check})
display(pd.DataFrame(rows))

## Cell 4: Zero-shot benchmarks (`src/bench_mc.py`): HellaSwag and ARC-Easy
Full validation/test sets, lm-evaluation-harness scoring. Random guessing is about 25%. The teacher should land near its published numbers (HellaSwag acc_norm about 54%); the 51M students are expected to sit only a few points above chance, so compare them using the standard errors.

In [ ]:
assert GPU, "Cell 4 needs a GPU runtime"
%cd /content/tensorix-kd-slm
for name, src in [(r, ["--ckpt", f"{CK}/{r}/best.pt"]) for r in runs] + [("teacher_base", ["--hf_model", TEACHER])]:
    if os.path.exists(f"{OUT}/bench/{name}.json") and not FORCE:
        print("already done:", name); continue
    run([sys.executable, "src/bench_mc.py", "--name", name, "--out_dir", f"{OUT}/bench"] + src)

rows = []
for f in sorted(glob.glob(f"{OUT}/bench/*.json")):
    j = json.load(open(f))
    rows.append({"model": j["name"], **{f"{t} acc_norm": f"{v['acc_norm']*100:.1f} +/- {v['acc_norm_stderr']*100:.1f}"
                                        for t, v in j["tasks"].items()}})
display(pd.DataFrame(rows))

## Cell 5: Held-out prompts and answers from every instruction-tuned model
The prompts come from the Dolly validation split used by all SFT runs (none of them was trained on these prompts); the script stops if the split differs from `split.json`. Every model gets the same prompt, the same decoding settings and the same random seed per prompt.

In [ ]:
assert GPU, "Cell 5 needs a GPU runtime"
%cd /content/tensorix-kd-slm
PROMPTS, GENS = f"{OUT}/dolly_eval_prompts.jsonl", f"{OUT}/generations.jsonl"
if not os.path.exists(PROMPTS) or FORCE:
    run([sys.executable, "src/make_dolly_eval.py", "--out", PROMPTS, "--n", str(N_PROMPTS),
         "--ref_split", SFT_REF_SPLIT])
if not os.path.exists(GENS) or FORCE:
    run([sys.executable, "experiments/gemma_sft/06_generate_eval.py", "--prompts", PROMPTS, "--out", GENS,
         "--temperature", str(GEN["temperature"]), "--top_k", str(GEN["top_k"]),
         "--max_new_tokens", str(GEN["max_new_tokens"]), "--seed", str(GEN["seed"]),
         "--models"] + [f"{k}={CK}/{v}/best.pt" for k, v in sft.items()])
else:
    print("already done:", GENS)

gens = [json.loads(l) for l in open(GENS, encoding="utf-8")]
for g in gens[:4]:
    print("\n" + "=" * 100 + f"\n[{g['category']}] {g['instruction'][:200]}")
    print("  [reference]", g["human_reference"][:200])
    for k, v in g["answers"].items():
        print(f"  [{k}] {v[:200]}")

## Cell 6: Blind pairwise judging (CPU runtime is fine)
Add your Gemini API key in Colab **Secrets** (key icon on the left) as `GEMINI_API_KEY` and allow notebook access. The key is never printed or saved. Prompts are from the public Dolly dataset; check the API's data-use terms anyway.
A model wins a prompt only if the judge picks it in **both** answer orders. Progress is saved after every call, so if the quota runs out, rerun this cell later (or the next day).

In [ ]:
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
%cd /content/tensorix-kd-slm
GENS = f"{OUT}/generations.jsonl"
for pair, n, forced in JUDGE_JOBS:
    x, y = pair.split(":")
    if x not in sft or y not in sft:
        print("skip (model missing):", pair); continue
    tag = f"{x}_vs_{y}_{'forced' if forced else 'ties'}_n{n}"
    print(f"\n===== {tag} =====")
    run([sys.executable, "experiments/gemma_sft/07_judge.py", "--gens", GENS, "--pairs", pair,
         "--judgments", f"{OUT}/judge/{tag}.jsonl", "--judge_model", JUDGE_MODEL, "--rpm", str(JUDGE_RPM),
         "--n", str(n), "--out_dir", f"{OUT}/judge"] + (["--forced"] if forced else []))

if HUMAN_SHEET:
    x, y = JUDGE_JOBS[0][0].split(":")
    run([sys.executable, "experiments/gemma_sft/07_judge.py", "--gens", GENS, "--pairs", f"{x}:{y}",
         "--export_human", str(HUMAN_SHEET), "--out_dir", f"{OUT}/judge"])
    print("Give judge/human_check.csv to 2-3 teammates (blind); keep judge/human_check_key.csv hidden.")

## Cell 7: Final report, figures and the zip for the repo
Writes `RESULTS.md`, `final_metrics.json` and `fig_*.png` to `results/final/` on Drive and builds `final_results.zip` (small files only, no checkpoints). Rerun whenever a new result arrives.

In [ ]:
%cd /content/tensorix-kd-slm
SFT_JSON = f"{DRIVE}/results/sft/sft_comparison.json"          # written by the Dolly SFT notebook (Section 13)
if os.path.exists(SFT_JSON):
    shutil.copy(SFT_JSON, f"{OUT}/sft_comparison.json")
judge_files = sorted(glob.glob(f"{OUT}/judge/*_summary.json"))
run([sys.executable, "src/final_report.py", "--eval_dir", f"{OUT}/eval", "--log_dir", f"{OUT}/logs",
     "--bench_dir", f"{OUT}/bench", "--gens", f"{OUT}/generations.jsonl",
     "--sft_json", f"{OUT}/sft_comparison.json", "--out", OUT, "--judge"] + judge_files)

from IPython.display import Image, display
for f in sorted(glob.glob(f"{OUT}/fig_*.png")):
    print(os.path.basename(f)); display(Image(f, width=900))

import zipfile
with zipfile.ZipFile("/content/final_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/**/*", recursive=True):
        if os.path.isfile(f) and not f.endswith("human_check_key.csv") and os.path.getsize(f) < 20e6:
            z.write(f, os.path.relpath(f, OUT))
shutil.copy("/content/final_results.zip", f"{OUT}/final_results.zip")
print("Download final_results.zip (Files panel, or Drive results/final/), unzip into results/final/ in the repo.")